# 2.1.5 — Dimensions, Shapes, Axes, and Array Semantics

**Module:** 02 — NumPy: Advanced Numerical Computing and Array Programming  
**Chapter:** 2.1 — NumPy Architecture and Fundamentals  
**Level:** Expert / Research-Oriented Foundations  
**Prerequisites:** Advanced Python object semantics; basic linear algebra; numerical analysis fundamentals.

> **Purpose.** Develop a precise, testable conceptual and computational model of dimensions, shapes, axes, and array semantics. This notebook separates mathematical intent, Python-level semantics, storage consequences, and execution behavior. It is a standalone section of the Chapter 2.1 architecture sequence.

## 1. Learning Objectives

After studying this notebook, you should be able to:

1. State the central abstractions and identify the level of the system at which each guarantee applies.
2. Explain the relevant mathematical or computational invariants without relying on informal analogies.
3. Construct executable tests that distinguish correct behavior from plausible but invalid assumptions.
4. Identify boundary cases, side effects, dtype concerns, and complexity implications.
5. Translate the topic into a reproducible, documented numerical research workflow.

## 2. Reproducibility and Environment

The examples use the standard `numpy` package and Python standard library. They are intended for recent NumPy 2.x installations; most examples also work on current 1.26 releases, but NumPy 2.x promotion and interface details should be interpreted with the installed version in mind. Execute cells **top to bottom**. There are no external datasets, downloads, or network dependencies.

In [ ]:
import numpy as np
import sys

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 24), 'Use a recent NumPy release.'

## 3. Conceptual Foundations and Formal Semantics

### 1. The index domain

If `A.shape == (n0, ..., nk-1)`, each axis defines a separate coordinate. A shape is an ordered tuple: `(T, N)` and `(N, T)` have different meanings despite the same element count. NumPy does not attach labels such as *time* or *asset* to axes; these must be documented and enforced by the caller.

### 2. Zero-dimensional and singleton axes

An array with `shape=()` is rank zero and contains one element. An axis of length one is present structurally and can affect broadcasting, reduction, and matrix multiplication. Squeezing changes structural semantics without necessarily changing scalar values.

### 3. Axis conventions

An axis argument selects a dimension for reduction or transformation; negative axis numbers count from the end. For `sum(axis=j)`, dimension `j` disappears unless `keepdims=True`. Keeping reduced dimensions often protects alignment in further operations.

### 4. Dimensional semantics and matrix algebra

NumPy's one-dimensional arrays do not intrinsically represent row or column vectors. `v.shape == (n,)` is different from `(1,n)` and `(n,1)`. `@` has explicit rules for 1-D and N-D operands; confusing batch axes and core matrix axes can silently produce the wrong calculation.

### 5. Shape contracts

Production numerical code should assert the expected semantic shape at API boundaries. A portfolio return tensor of shape `(T, N)` and weights `(N,)` supports a reduction across the asset dimension. Reversing axes or transposing without adjusting the contract yields results that may appear plausible while encoding a different model.

## 4. Architectural Model and Invariants

A useful four-layer checklist for this topic:

| Layer | Question to ask | Representative invariant |
|---|---|---|
| Logical | What operation is mathematically intended? | Axes and result shape correspond to the model |
| Representation | What dtype and storage model represent the inputs? | No unsupported casting or unexpected aliasing |
| Execution | Which category of operation processes the arrays? | Results are stable under a reference implementation |
| Validation | What independent evidence establishes correctness? | Assertions, boundary tests, and numerical tolerances |

**Important:** Shape agreement and code execution are not sufficient evidence of mathematical validity. Conversely, a mathematically valid expression can be operationally impractical or numerically unstable.

## 5. Executable Investigations

Each investigation is independent in purpose; execute in sequence so shared imports are available.

### 5.1 — Axis reduction and keepdims

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
import numpy as np
x = np.arange(24).reshape(2, 3, 4)
y = x.sum(axis=1)
z = x.sum(axis=1, keepdims=True)
assert y.shape == (2, 4)
assert z.shape == (2, 1, 4)
assert np.array_equal(z[:,0,:], y)

### 5.2 — Row/column/vector distinctions

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
v = np.array([1., 2., 3.])
row = v[None, :]
col = v[:, None]
assert v.shape == (3,)
assert row.shape == (1, 3) and col.shape == (3, 1)
assert (col @ row).shape == (3, 3)
assert (row @ col).shape == (1, 1)

### 5.3 — Semantic axis contracts in returns

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
r = np.array([[0.01, -0.02, 0.03], [0.02, 0.01, -0.01]])  # (time, assets)
w = np.array([0.3, 0.4, 0.3])  # (assets,)
assert r.ndim == 2 and r.shape[1] == w.shape[0]
p = r @ w
print('portfolio returns by time:', p)
assert p.shape == (2,)

## 6. Edge Cases, Failure Modes, and Misconceptions

- **Logical validity is not physical contiguity.** Shape alone cannot determine whether an operation needs a copy.
- **Value equality is not identity.** Two arrays may compare equal while having independent storage, or share storage while being distinct Python objects.
- **A vectorized expression does not certify correctness.** Broadcasting and axis conventions can silently encode unintended mathematics.
- **A dtype is not a real-number model.** Overflow, finite precision, and representation-dependent transformations require explicit attention.
- **A benchmark does not imply a theorem.** Hardware, build configuration, allocation strategy, and input scale can affect conclusions.
- **Array semantics do not encode domain assumptions.** For finance or statistics, look-ahead restrictions, missing-data rules, weights, and alignment remain external responsibilities.

When evaluating an unexpected result, first inspect `shape`, `dtype`, `strides`, and storage overlap, then isolate the mathematical operation on a hand-checkable example.

## 7. Performance and Complexity Considerations

The performance question should be asked in three parts: **work complexity**, **data movement**, and **peak live memory**. A contiguous numeric array can support efficient native processing, but a particular operation may be limited by memory bandwidth, temporary allocation, scalar dispatch, or native-library parallelism. Use measurements only after verifying correctness, and never assume that a microbenchmark generalizes to different dtypes or problem sizes.

A practical benchmark should (i) isolate the operation, (ii) distinguish input setup from computation, (iii) warm up the relevant execution path, (iv) collect repeated measurements, and (v) record platform and NumPy provenance.

## 8. Analytical Interpretation

The **meaning** of an axis is an external invariant; NumPy only knows its length and position. Shape-aware programming is therefore a form of lightweight dimensional analysis. Document named conceptual dimensions even when using unlabeled raw arrays.

## 9. Advanced Exercises

These are open-ended expert exercises. Provide an analytical justification, a minimal counterexample where useful, and an executable validation test.

**Exercise 1.** Specify a complete shape contract for an array `(simulation, time, asset, factor)`.

**Exercise 2.** Determine the output shapes for multiple `sum(axis=..., keepdims=...)` operations without executing code.

**Exercise 3.** Explain why `np.squeeze` can introduce subtle shape bugs when a batch dimension happens to be 1.

## 10. Expected Results and Validation Strategy

1. Execute every example in order; all assertions should pass on an appropriately configured NumPy installation.
2. Compare printed shapes, types, and DTypes with the surrounding mathematical and architectural claims.
3. Where a local benchmark is included, record measured values but **do not** expect a universal numeric timing.
4. Modify input shapes, dtypes, and views to construct at least one failing case; document why the original contract no longer holds.
5. Implement an independent reference for one operation and compare using an error metric appropriate to its numerical domain.

**Acceptance criteria:** the student can explain the behavior without relying on a copied output, design a counterexample to an invalid inference, and describe relevant representation and performance limitations.

## Further Reading and Verification Sources

The following are authoritative public references. API details may vary across NumPy versions; verify against the installed version when reproducing an experiment.

- [NumPy: The N-dimensional array](https://numpy.org/doc/stable/reference/arrays.ndarray.html)
- [NumPy: ndarray reference](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)
- [NumPy: Data types](https://numpy.org/doc/stable/reference/arrays.dtypes.html)
- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy: Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [NumPy: Universal functions](https://numpy.org/doc/stable/reference/ufuncs.html)
- [NumPy: Thread safety](https://numpy.org/doc/stable/reference/thread_safety.html)
- [NumPy: C API](https://numpy.org/doc/stable/reference/c-api/index.html)

**Scope note:** This is Chapter 2.1 (architectural foundations). Detailed DType implementation, advanced indexing, stride tricks, microarchitecture, and native C extension development are treated in Chapters 2.3, 2.5, 2.7, 2.21 and 2.23 respectively.